In [22]:
import numpy as np 
import pandas as pd  

In [23]:
import sys
from pathlib import Path
import shutil

code_dir = Path("/kaggle/working/code")

# If the folder already exists, remove it so we can do a fresh clone
if code_dir.exists():
    shutil.rmtree(code_dir)

# Clone your project repository into Kaggle's working directory
!git clone https://github.com/NaomiMeseret/amharic-ngram-language-model /kaggle/working/code

# Add the cloned repository to Python's search path
sys.path.insert(0, "/kaggle/working/code")

print("Repository cloned and path configured successfully!")

Cloning into '/kaggle/working/code'...
remote: Enumerating objects: 357, done.
remote: Counting objects: 100% (357/357), done.
remote: Compressing objects: 100% (244/244), done.
remote: Total 357 (delta 140), reused 267 (delta 105), pack-reused 0 (from 0)
Receiving objects: 100% (357/357), 4.21 MiB | 22.83 MiB/s, done.
Resolving deltas: 100% (140/140), done.
Repository cloned and path configured successfully!


In [24]:
!grep -rn "predict_next" /kaggle/working/code --include=*.py

/kaggle/working/code/src/ngram_model.py:472:    def predict_next(self, context_words: str | Sequence[str], k: int = 5,
/kaggle/working/code/src/train_ngram_slice.py:214:    predictions_before = model.predict_next("ሐምሌ absentword", k=5)
/kaggle/working/code/src/train_ngram_slice.py:215:    predictions_after = loaded.predict_next("ሐምሌ absentword", k=5)
/kaggle/working/code/src/train_ngram_slice.py:305:        "predict_next_full_distribution_note": (
/kaggle/working/code/src/preprocess_lib.py:1:"""Shared text rules for GPAC preprocessing, predict_next() and the app.
/kaggle/working/code/tests/test_ngram_model.py:135:        predictions = self.model.predict_next("እኔ", k=5)
/kaggle/working/code/tests/test_ngram_model.py:139:        unigram_predictions = self.model.predict_next([], k=5)
/kaggle/working/code/tests/test_ngram_model.py:142:        self.assertEqual(self.model.predict_next("", k=0), [])
/kaggle/working/code/tests/test_ngram_model.py:143:        self.assertEqual(self.model.predict

In [25]:
import zipfile
from pathlib import Path
from src.ngram_model import NGramModel

# Locate and extract the trained model zip from Kaggle inputs
input_dir = Path("/kaggle/input")
zip_path = next(input_dir.rglob("trained_model.zip"), None)

if zip_path and zip_path.is_file():
    print(f"Found trained model archive at: {zip_path}")
    target_dir = Path("/kaggle/working/ngram_results")
    target_dir.mkdir(parents=True, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(target_dir)
    print("Model extracted successfully!")
else:
    print("Model archive not found. Make sure Naomi's notebook output is added as input.")

# Load the model
model_dir = Path("/kaggle/working/ngram_results/model")
model = NGramModel.load(model_dir, mmap_mode="r")
print(f"Successfully loaded model with max order: {model.max_order}")

Found trained model archive at: /kaggle/input/notebooks/naomimeseret/amharic-n-gram-model-building-and-training-membe/ngram_results/final_downloads/trained_model.zip
Model extracted successfully!
Successfully loaded model with max order: 4


In [26]:
import time
from src.ngram_model import BOS, UNK, normalize_and_tokenize

V, W2I = model.vocabulary, model.word_to_id

print(repr(V[UNK]), normalize_and_tokenize(V[UNK]))   # must NOT be an empty list
a = model.predict_next(["አዲስ", "አበባ", "የኢትዮጵያ", "ዋና"], k=5)
b = model.predict_next(["አበባ", "የኢትዮጵያ", "ዋና"], k=5)
print("only last 3 words matter:", a == b)
t0 = time.time(); model.predict_next(["አዲስ"], k=5); dt = time.time() - t0
print(f"{dt:.1f}s per call -> worst case {26173 * dt / 3600:.1f} hours for 26,173 positions")

'<UNK>' ['UNK']
only last 3 words matter: True
2.2s per call -> worst case 15.8 hours for 26,173 positions


In [27]:
from src.ngram_model import normalize_and_tokenize
import time

print(repr(V[UNK]), normalize_and_tokenize(V[UNK]))   # must NOT be an empty list
a = model.predict_next(["አዲስ", "አበባ", "የኢትዮጵያ", "ዋና"], k=5)
b = model.predict_next(["አበባ", "የኢትዮጵያ", "ዋና"], k=5)
print("only last 3 words matter:", a == b)
t0 = time.time(); model.predict_next(["አዲስ"], k=5)
print(f"{time.time() - t0:.1f}s per call -> worst case {26173 * (time.time() - t0) / 3600:.1f} hours for 26,173 positions")

'<UNK>' ['UNK']
only last 3 words matter: True
2.0s per call -> worst case 14.7 hours for 26,173 positions


In [28]:
from pathlib import Path
import numpy as np

# Search the entire Kaggle directory for test.npy
all_tests = list(Path("/kaggle").rglob("test.npy"))

if all_tests:
    test_path = all_tests[0]
    print(f"Found test.npy at: {test_path}")
    
    test_ids = np.load(test_path, mmap_mode="r", allow_pickle=False)
    print(f"Successfully loaded test array with shape: {test_ids.shape}")
else:
    print("test.npy not found yet. Check the left-hand panel to ensure the upload progress ring has finished spinning!")

Found test.npy at: /kaggle/input/datasets/rahelyabebal/test-data/test.npy
Successfully loaded test array with shape: (8579227,)


### Task 3: Extrinsic Evaluation 

We evaluate model utility on two core downstream tasks:

Next-Word Prediction (Autocomplete): Measures how accurately the model predicts the upcoming token given a preceding context window, reported via Top-1, Top-3, and Top-5 accuracy.

Real-vs-Shuffled Sentence Discrimination (Fluency): Evaluates whether the model assigns higher aggregate log-probabilities to natural Amharic sentences versus scrambled word-order equivalents, serving as a baseline for autocorrect and fluency scoring.

In [31]:
import random, time
import numpy as np
import pandas as pd
from src.ngram_model import iter_encoded_sentences
import src.extrinsic_eval as ext

# 1. Decode the flat ID stream into list-of-int sentences (validates BOS/EOS pairing)
all_sentences = [s for s in iter_encoded_sentences(test_ids) if len(s) >= 5]
print(f"{len(all_sentences):,} usable test sentences in total")

# 2. Random samples (same seed and same call order as before, so same samples)
rnd = random.Random(42)
nw_sample   = rnd.sample(all_sentences, min(2000, len(all_sentences)))
disc_sample = rnd.sample(all_sentences, min(1000, len(all_sentences)))

# 3. Next-word prediction (exact-order, no backoff)
nw_results = []
for n in range(1, 5):
    t0 = time.time()
    res = ext.next_word_eval(model, nw_sample, order=n, ks=(1, 3, 5))
    res["order"] = n
    nw_results.append(res)
    print(f"next-word order {n} done in {time.time() - t0:.0f}s")

df_nw = pd.DataFrame(nw_results)
assert df_nw["evaluated"].nunique() == 1, "orders scored different positions"
print("\n--- Next-Word Prediction Benchmark ---")
print(df_nw[["order", "top1", "top3", "top5", "top1_given_seen", "top5_given_seen",
             "context_seen", "has_suggestion", "evaluated", "skipped_unk", "skipped_eos"]]
      .to_string(index=False))

# 4. Real vs. shuffled discrimination (backoff-smoothed log-probabilities)
disc_results = []
for n in range(1, 5):
    t0 = time.time()
    res = ext.discrimination_eval(model, disc_sample, order=n, method="backoff")
    res["order"] = n
    disc_results.append(res)
    print(f"discrimination order {n} done in {time.time() - t0:.0f}s")

df_disc = pd.DataFrame(disc_results)
print("\n--- Real vs. Shuffled Sentence Discrimination ---")
print(df_disc[["order", "accuracy", "strict_accuracy", "ties", "wins", "n", "skipped"]]
      .to_string(index=False))

489,374 usable test sentences in total
next-word order 1 done in 0s
next-word order 2 done in 433s
next-word order 3 done in 155s
next-word order 4 done in 113s

--- Next-Word Prediction Benchmark ---
 order     top1     top3     top5  top1_given_seen  top5_given_seen  context_seen  has_suggestion  evaluated  skipped_unk  skipped_eos
     1 0.017499 0.044473 0.054636         0.017499         0.054636      1.000000        1.000000      26173         4189         2000
     2 0.108012 0.181905 0.220838         0.108012         0.220838      1.000000        1.000000      26173         4189         2000
     3 0.131662 0.196577 0.225041         0.152667         0.260943      0.862415        0.854621      26173         4189         2000
     4 0.114240 0.149849 0.163031         0.206463         0.294642      0.553318        0.541512      26173         4189         2000
discrimination order 1 done in 1s
discrimination order 2 done in 158s
discrimination order 3 done in 273s
discrimination ord

In [32]:
import os
from pathlib import Path
out = Path("/kaggle/working/results"); out.mkdir(parents=True, exist_ok=True)
df_nw.assign(sample_sentences=len(nw_sample)).to_csv(out / "extrinsic_next_word.csv", index=False)
df_disc.assign(sample_sentences=len(disc_sample)).to_csv(out / "extrinsic_discrimination.csv", index=False)
!git -C /kaggle/working/code rev-parse HEAD

5f7ea0af0958fe6bdc758615e8e5992f17000501


In [33]:
import math, random
from src.ngram_model import BOS, EOS, UNK, normalize_and_tokenize

# ---------- edit these ----------
CUSTOM = [
    "ኢትዮጵያ በአፍሪካ ቀንድ ትገኛለች።",
    "አዲስ አበባ የኢትዮጵያ ዋና ከተማ ናት።",
]                      # replace with your own sentences
N_FROM_TEST = 2        # extra random real sentences from the test set
ORDERS = (2, 3, 4)
# --------------------------------

V, W2I = model.vocabulary, model.word_to_id
SPECIAL = {0, 1, 2, 3}

def logp(ids, order):
    """log P(sentence) with backoff, </s> included, same framing as training."""
    framed = [BOS] * (order - 1) + ids + [EOS]
    return sum(model.logprob(framed[i], framed[i - order + 1:i], method="backoff")
               for i in range(order - 1, len(framed)))

def exact_suggestions(prefix_ids, order, k=5):
    ctx = ([BOS] * (order - 1) + prefix_ids)[-(order - 1):]
    pairs = [(t, c) for t, c in model.orders[order].next_counts(ctx) if t not in SPECIAL]
    pairs.sort(key=lambda p: -p[1])
    return [V[t] for t, _ in pairs[:k]]

# collect sentences as word lists
items = [normalize_and_tokenize(t) for t in CUSTOM]
if N_FROM_TEST and "all_sentences" in globals():
    rnd = random.Random(5)
    pool = [s for s in all_sentences if 8 <= len(s) <= 12 and UNK not in s]
    items += [[V[t] for t in s[1:-1]] for s in rnd.sample(pool, N_FROM_TEST)]

rng = random.Random(1)
for words in items:
    ids = [W2I.get(w, UNK) for w in words]
    shuffled = ids[:]
    for _ in range(20):
        rng.shuffle(shuffled)
        if shuffled != ids: break

    print("=" * 70)
    print("REAL    :", " ".join(words))
    print("SHUFFLED:", " ".join(V[i] for i in shuffled))

    print("\nTest 1: which is more likely? (higher log-prob wins)")
    for n in ORDERS:
        a, b = logp(ids, n), logp(shuffled, n)
        print(f"  n={n}: real {a:8.1f} | shuffled {b:8.1f} -> {'real wins ✓' if a > b else 'WRONG ✗'}")

    print("\nTest 2: next-word suggestions (exact n-gram, no backoff)")
    for i in range(1, len(ids)):
        prefix = ids[:i]
        line = [f"n={n} " + ("✓" if words[i] in (s := exact_suggestions(prefix, n)) else "✗") for n in ORDERS]
        print(f"  {' '.join(words[:i]):<35} -> actual: {words[i]:<12} {'  '.join(line)}")
        print(f"      n=3 suggests: {', '.join(exact_suggestions(prefix, 3)) or '(context unseen)'}")

REAL    : ኢትዮጵያ በአፍሪካ ቀንድ ትገኛለች
SHUFFLED: ትገኛለች ኢትዮጵያ ቀንድ በአፍሪካ

Test 1: which is more likely? (higher log-prob wins)
  n=2: real    -25.9 | shuffled    -55.8 -> real wins ✓
  n=3: real    -25.1 | shuffled    -57.2 -> real wins ✓
  n=4: real    -25.6 | shuffled    -57.2 -> real wins ✓

Test 2: next-word suggestions (exact n-gram, no backoff)
  ኢትዮጵያ                               -> actual: በአፍሪካ        n=2 ✗  n=3 ✗  n=4 ✗
      n=3 suggests: ውስጥ, ነገ, በክብር, ለዘለአለም, በአለም
  ኢትዮጵያ በአፍሪካ                         -> actual: ቀንድ          n=2 ✓  n=3 ✓  n=4 ✓
      n=3 suggests: ቀንድ, ውስጥ, ደረጃ, አህጉር, ብሎም
  ኢትዮጵያ በአፍሪካ ቀንድ                     -> actual: ትገኛለች        n=2 ✗  n=3 ✗  n=4 ✗
      n=3 suggests: አካባቢ, ላይ, ውስጥ, ሰላምና, ሰላም
REAL    : አዲስ አበባ የኢትዮጵያ ዋና ከተማ ናት
SHUFFLED: ከተማ ናት አበባ ዋና አዲስ የኢትዮጵያ

Test 1: which is more likely? (higher log-prob wins)
  n=2: real    -29.0 | shuffled    -58.7 -> real wins ✓
  n=3: real    -19.5 | shuffled    -65.6 -> real wins ✓
  n=4: real    -16.3 | shuffled    -

In [34]:
import os, time
import multiprocessing as mp
from src.ngram_model import BOS, UNK, normalize_and_tokenize

V, W2I = model.vocabulary, model.word_to_id
PAD_N = model.max_order - 1
print("'UNK' in vocab:", "UNK" in W2I)                  # should be False

N = 150                                                 # lower this if the estimate is too long
sub = nw_sample[:N]

def ctx_key(prefix):
    return tuple(t for t in prefix if t != BOS)[-PAD_N:]

ctxs = set()
for sent in sub:
    padded = [BOS] * PAD_N + sent[1:]
    for i in range(PAD_N, len(padded)):
        if padded[i] not in ext.SPECIAL:                # same exclusions as the evaluator
            ctxs.add(ctx_key(padded[:i]))
cores = os.cpu_count()
print(f"{len(ctxs)} unique contexts, {cores} cores, est. ~{len(ctxs) * 2.0 / cores / 60:.0f} min")

'UNK' in vocab: False
1956 unique contexts, 4 cores, est. ~16 min


In [37]:
import os, time
import multiprocessing as mp
import pandas as pd
from pathlib import Path
from src.ngram_model import BOS
import src.extrinsic_eval as ext

V, W2I = model.vocabulary, model.word_to_id
PAD_N = model.max_order - 1
cores = os.cpu_count()
out = Path("/kaggle/working/results"); out.mkdir(parents=True, exist_ok=True)

BUDGET = 200            # max unique contexts to run through predict_next (raise it if this is fast)
USE_POOL = True         # set False if the pool hangs; slower (single process) but safe

def ctx_key(prefix):
    return tuple(t for t in prefix if t != BOS)[-PAD_N:]

def _pred(ctx):
    return ctx, [W2I[w] for w, _ in model.predict_next([V[t] for t in ctx], k=5)]

# 1. Add sentences until the unique-context budget is reached
sub, ctxs = [], set()
for sent in nw_sample:
    padded = [BOS] * PAD_N + sent[1:]
    new = {ctx_key(padded[:i]) for i in range(PAD_N, len(padded))
           if padded[i] not in ext.SPECIAL}
    if len(ctxs | new) > BUDGET and sub:
        break
    sub.append(sent); ctxs |= new
print(f"{len(sub)} sentences, {len(ctxs)} unique contexts")

# 2. Run the real predict_next once per unique context, printing progress
cache, t0 = {}, time.time()
if USE_POOL:
    with mp.get_context("fork").Pool(cores) as pool:
        for j, (c, ids) in enumerate(pool.imap_unordered(_pred, ctxs, chunksize=2), 1):
            cache[c] = ids
            if j % 20 == 0:
                print(f"{j}/{len(ctxs)}  {(time.time() - t0) / 60:.1f} min", flush=True)
else:
    for j, c in enumerate(ctxs, 1):
        cache[c] = _pred(c)[1]
        if j % 10 == 0:
            print(f"{j}/{len(ctxs)}  {(time.time() - t0) / 60:.1f} min", flush=True)
print(f"predict_next done in {(time.time() - t0) / 60:.1f} min")

# 3. Evaluate every row on the same sentences
def backoff_predictor(context_ids, k):
    ids = cache[ctx_key(context_ids)]       # KeyError means a context was missed
    return ids[:k], bool(ids)

rows = []
for n in range(1, 5):
    r = ext.next_word_eval(model, sub, order=n); r["order"] = n; rows.append(r)
r = ext.next_word_eval(model, sub, predictor=backoff_predictor, ks=(1, 3, 5))
r["order"] = "backoff"; rows.append(r)

df_sub = pd.DataFrame(rows)
assert df_sub["evaluated"].nunique() == 1, "rows scored different positions"
print(df_sub[["order", "top1", "top3", "top5", "has_suggestion", "evaluated"]].to_string(index=False))
df_sub.assign(sample_sentences=len(sub)).to_csv(out / "extrinsic_next_word_backoff_subsample.csv", index=False)

# 4. Proof the adapter equals the app's own call
c = next(iter(cache))
print("adapter == predict_next:", [V[i] for i in cache[c]] == [w for w, _ in model.predict_next([V[t] for t in c], k=5)])

11 sentences, 185 unique contexts
20/185  0.7 min
40/185  1.2 min
60/185  1.8 min
80/185  2.3 min
100/185  2.9 min
120/185  3.4 min
140/185  4.1 min
160/185  4.7 min
180/185  5.3 min
predict_next done in 5.4 min
  order     top1     top3     top5  has_suggestion  evaluated
      1 0.015228 0.055838 0.055838        1.000000        197
      2 0.116751 0.197970 0.233503        1.000000        197
      3 0.106599 0.197970 0.213198        0.827411        197
      4 0.106599 0.142132 0.147208        0.517766        197
backoff 0.147208 0.228426 0.258883        1.000000        197
adapter == predict_next: True


In [38]:
direct = [w for w, _ in model.predict_next(["አዲስ"], k=5)]
ids, _ = backoff_predictor([BOS, W2I["አዲስ"]], 5) if (ctx_key([BOS, W2I["አዲስ"]]) in cache) else (None, None)
if ids is not None:
    print("matches direct call:", [V[i] for i in ids] == direct)
else:
    print("that context isn't in the cache; compare against any key from `cache` instead")

bad = [w for w in V if w not in ("<PAD>", "<UNK>", "<s>", "</s>") and normalize_and_tokenize(w) != [w]]
print(len(bad), "vocab words change under normalize_and_tokenize:", bad[:10])

that context isn't in the cache; compare against any key from `cache` instead
0 vocab words change under normalize_and_tokenize: []
